# Project - Evaluate Models on FPN Search using Azure API
Searching for a good cheap model for FPNsearch Note the Azure data is not in RAG form

Resources:
* [Azure AI Search client library for Python - version 11.6.0](https://learn.microsoft.com/en-us/python/api/overview/azure/search-documents-readme?view=azure-python)

In [1]:
import os
import json
import random

from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

from datetime import datetime
import re
from huggingface_hub import HfApi, CommitOperationAdd
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient

import subprocess
from IPython.display import Markdown, display



In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:4]}")
else:
    print("OpenRouter API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AI
Grok API Key exists and begins xai-
Groq API Key exists and begins gsk_
OpenRouter API Key exists and begins sk-o


In [3]:
search_client = None
openai = OpenAI(api_key=openai_api_key)


anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
grok_url = "https://api.x.ai/v1"
groq_url = "https://api.groq.com/openai/v1"
ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)










In [4]:
# OPENAI_MODEL = "gpt-5"
# CLAUDE_MODEL = "claude-sonnet-4-5-20250929"
# GROK_MODEL = "grok-4"
# GEMINI_MODEL = "gemini-2.5-pro"

# Want to keep costs ultra-low? Uncomment these lines:

# OPENAI_MODEL = "gpt-5-nano"
# CLAUDE_MODEL = "claude-3-5-haiku-latest"
# GROK_MODEL = "grok-4-fast-non-reasoning"
# GEMINI_MODEL = "gemini-2.5-flash-lite"





#tried models that did not work well at all:
#"gemma-4-26b-a4b-it": gemini, "gemma-4-31b-it": gemini


#models that work:
#"claude-haiku-4-5": anthropic,
#"gemini-3.1-flash-lite-preview": gemini

#trying to get this model to work - but cannot be found: "gpt-oss-120b": openai,,"gemma-4-31B": gemini,

CLIENTS = {
    "openai/gpt-oss-120b": openrouter,
    "ai21/jamba-large-1.7":openrouter,
    "amazon/nova-pro-v1":openrouter,
    "meta-llama/llama-3.3-70b-instruct":openrouter,
    "minimax/minimax-m2.5":openrouter,
    "moonshotai/kimi-k2-thinking":openrouter
    } 
MODELS= list(CLIENTS.keys())
print(MODELS)

EVAL_CLIENTS = {"gpt-4.1-mini":openai} #,"claude-sonnet-4-5":anthropic
EVAL_MODELS= list(EVAL_CLIENTS.keys())



['openai/gpt-oss-120b', 'ai21/jamba-large-1.7', 'amazon/nova-pro-v1', 'meta-llama/llama-3.3-70b-instruct', 'minimax/minimax-m2.5', 'moonshotai/kimi-k2-thinking']


In [ ]:
#MODELS= ['gpt-4.1-mini']

# MODELS = ['gpt-oss-120b','google/gemma-4-31B-it']



In [5]:


# Initialization
def GetSearchClient():


    azure_search_api_key = os.getenv('AZURE_SEARCH_API_KEY')
    if azure_search_api_key:
        print(f"AZURE_SEARCH_API_KEY: {azure_search_api_key[:8]}...")
    else:
        print("AZURE_SEARCH_API_KEY not set")


    azure_search_service_endpoint = os.getenv('AZURE_SEARCH_SERVICE_ENDPOINT')
    if azure_search_service_endpoint:
        print(f"AZURE_SEARCH_SERVICE_ENDPOINT: {azure_search_service_endpoint}")
    else:
        print("AZURE_SEARCH_SERVICE_ENDPOINT not set")


    azure_search_index_name = os.getenv('AZURE_SEARCH_INDEX_NAME')
    if azure_search_index_name:
        print(f"AZURE_SEARCH_INDEX_NAME: {azure_search_index_name}")
    else:
        print("AZURE_SEARCH_INDEX_NAME not set")


    service_endpoint = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
    index_name = os.environ["AZURE_SEARCH_INDEX_NAME"]
    key = os.environ["AZURE_SEARCH_API_KEY"]

    return SearchClient(azure_search_service_endpoint, azure_search_index_name, AzureKeyCredential(azure_search_api_key))



In [6]:
def azure_search(query, nTopResults = 5):
    print(f"Azure Search Tool called for query: {query}")
    results = search_client.search(search_text=query, top=nTopResults)
    return results

In [ ]:
# def GetOpenAIKey():
#     openai_api_key = os.getenv('OPENAI_API_KEY')

#     if openai_api_key:
#         print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
#     else:
#         print("OpenAI API Key not set")
#     return openai_api_key


In [7]:
with open("system_question.txt", "r") as f:
    system_message = f.read()

with open("system_evaluator.md", "r") as f:
    system_message_evaluator = f.read()


In [8]:
def get_references(json_data,root_path="https://fpnotebook.com/"):
    references = []
    for result in json_data:
        content = result['content']
        page_url = content['PageUrl']
        title = content['Title']
        references.append(f"[{title}]({root_path}{page_url}.htm)")
    return  "\n\n**References from FPnotebook:**\n" + "\n".join(references)

In [9]:
include_history = False
def chat(message, history, search_result, model, client):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    if not include_history:
        history = []        
    
    message_new = f"""User question: {message}\n\n
    Related Information from FPNotebook in JSON: {search_result}"""
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message_new}]

    response = client.chat.completions.create(model=model, messages=messages)
    return response.choices[0].message.content

In [10]:
def evaluate_answer(question, answer, model, client):
    prompt = f"""You are an expert evaluator of question-answering. Please evaluate the quality of the answer based on the question. Here is the question: {question} and here is the answer: {answer}."""
    messages = [{"role": "system", "content": system_message_evaluator}, {"role": "user", "content": prompt}]
    response = client.chat.completions.create(model=model, messages=messages, response_format={"type": "json_object"})
    result = response.choices[0].message.content
    eval_json = json.loads(result)
    return eval_json

In [11]:
search_client = GetSearchClient()
#openai_api_key = GetOpenAIKey()

AZURE_SEARCH_API_KEY: dMjWKzaj...
AZURE_SEARCH_SERVICE_ENDPOINT: https://fpnsearchfortypingmind.search.windows.net
AZURE_SEARCH_INDEX_NAME: azureblob-index2


In [ ]:
questionID = "FPN_002"
question = "Describe the initial management steps for a patient presenting with acute ischemic stroke."
search_result = azure_search(question)
references = get_references(search_result)

data = {}
data['question'] = question
data['references'] = references
data['models']={}
for model in MODELS:
    print(f"Testing model: {model}")
    answer = chat(question, [], search_result, model, CLIENTS[model])
    print(f"Answer from {model}: {answer}")
    data['models'][model] = {}
    data['models'][model]['answer'] = answer
    data['models'][model]['evals'] = {}
    for eval_model in EVAL_MODELS:
        eval_json = evaluate_answer(question, answer, eval_model, EVAL_CLIENTS[eval_model])
        print(f"Evaluation of {model} by {eval_model}: {eval_json}")
        data['models'][model]['evals'][eval_model] = eval_json

with open(questionID + '.json', 'w') as outfile:
    json.dump(data, outfile)

Azure Search Tool called for query: What are the key features that distinguish MI chest pain from other causes?
Testing model: openai/gpt-oss-120b
Answer from openai/gpt-oss-120b: **Key distinguishing features of myocardial‑infarction (MI) chest pain (as outlined on the FPNotebook page)**  

MI pain is classically described as a **“crushing” or “pressure‑like”** sensation that is **more intense than typical angina**, persists **>30 minutes**, and is **not fully relieved by rest, nitroglycerin, or sublingual nitrates (even after three doses)**. The discomfort often **radiates** atypically—most suggestive when it spreads to the **right arm, both arms, shoulder, jaw, neck, upper back, or throat**—and is accompanied by **autonomic signs** such as **profuse diaphoresis, nausea/vomiting, and a profound sense of impending doom**. Unlike musculoskeletal or reflux pain, MI pain **does not vary with respiration or palpation** and is frequently **exertion‑related** (onset with activity, relieved 